# Demo 10b — $k$-means and hierarchical clustering

**Class 10 · B1** (slides: `slides/10_clustering.md`)

* Lloyd's algorithm step by step, and the cost $J = \sum_i \lVert x_i - \mu_{z_i} \rVert^2$ that never rises.
* Random start against $k$-means++: spread of the cost over 200 seeds.
* Agglomerative clustering: the four linkages, the dendrogram and the cut.
* Determinism: hierarchical clustering gives the same tree on every run, unseeded $k$-means does not.
* Equivalences: Ward against $k$-means, single linkage against DBSCAN, and where each one breaks.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from scipy.cluster import hierarchy as hc
from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import adjusted_rand_score as ari

from faa import clustering as cl
from faa import demo, plotlib
from faa.demo import show, slide

demo.setup("10")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)
COLS = [BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY]
SEED = 0
B = cl.benchmarks(SEED)

## Slide: Lloyd's algorithm on ten points

The running example of the class: ten points in the plane, $k = 2$. Assign every point to the nearest centre,
move every centre to the mean of its points, repeat until nothing changes.

In [ ]:
P = np.array(
    [
        [1.0, 1.0],
        [1.5, 2.0],
        [2.0, 1.5],
        [1.0, 2.5],
        [2.5, 2.0],
        [6.0, 5.0],
        [7.0, 6.0],
        [6.5, 7.0],
        [7.5, 5.5],
        [8.0, 7.0],
    ]
)
mu = P[[0, 5]].copy()
hist = []
for it in range(6):
    D2 = ((P[:, None] - mu[None]) ** 2).sum(-1)
    z = D2.argmin(1)
    J = D2[np.arange(len(P)), z].sum()
    hist.append((it, J, mu.copy(), z.copy()))
    new = np.array([P[z == j].mean(0) for j in range(2)])
    if np.allclose(new, mu):
        break
    mu = new
slide("Lloyd's algorithm on ten points")
show(
    pl.DataFrame(
        [(i, J, str(m.round(2).tolist())) for i, J, m, _ in hist], schema=["iteration", "J", "centres"], orient="row"
    )
)
f, ax = plt.subplots(1, len(hist[:3]), figsize=(plotlib.FULL, 1.45), sharex=True, sharey=True)
for a, (i, J, m, zz) in zip(ax, hist[:3]):
    a.scatter(*P.T, c=[COLS[t] for t in zz], s=14)
    a.scatter(*m.T, marker="X", c="k", s=40)
    a.set_title(f"iteration {i}: J = {J:.1f}")
plotlib.save(f, "b1_lloyd")

## Slide: Random start against $k$-means++

Initial centres from the data points at random, against $k$-means++ (the next centre is drawn with probability
proportional to the squared distance to the nearest chosen centre). One run each (`n_init=1`), 200 seeds.

In [ ]:
X, y = B["varied"]
rows = []
for init in ("random", "k-means++"):
    inertia = np.array([KMeans(3, init=init, n_init=1, random_state=s).fit(X).inertia_ for s in range(200)])
    rows.append((init, inertia.min(), np.median(inertia), inertia.max(), (inertia > inertia.min() * 1.05).mean()))
slide("Random start against k-means++")
show(pl.DataFrame(rows, schema=["init", "best J", "median J", "worst J", "share > 5% above best"], orient="row"))
f, ax = plt.subplots(figsize=(plotlib.COL, 1.5))
for init, c in zip(("random", "k-means++"), (ORANGE, BLUE)):
    v = [KMeans(3, init=init, n_init=1, random_state=s).fit(X).inertia_ for s in range(200)]
    ax.hist(v, bins=20, alpha=0.7, color=c, label=init)
ax.set_xlabel("final cost $J$"), ax.set_ylabel("runs"), ax.legend()
plotlib.save(f, "b1_init")

## Slide: Voronoi cells and the cost of a wrong $k$

$k$-means partitions the plane into the Voronoi cells of its centres: the boundary is a straight line halfway
between two centres. It cuts a long cluster in two if $k$ is too large.

In [ ]:
Xb, yb = B["blobs"]
f, ax = plt.subplots(1, 3, figsize=(plotlib.FULL, 1.45), sharex=True, sharey=True)
gx, gy = np.meshgrid(
    np.linspace(Xb[:, 0].min() - 1, Xb[:, 0].max() + 1, 200), np.linspace(Xb[:, 1].min() - 1, Xb[:, 1].max() + 1, 200)
)
for a, k in zip(ax, (2, 3, 6)):
    m = KMeans(k, init="k-means++", n_init=32, random_state=SEED).fit(Xb)
    a.contourf(
        gx,
        gy,
        m.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape),
        alpha=0.2,
        colors=COLS[:k],
        levels=np.arange(k + 1) - 0.5,
    )
    a.scatter(*Xb.T, c=[COLS[t] for t in m.labels_], s=5, lw=0)
    a.scatter(*m.cluster_centers_.T, marker="X", c="k", s=30)
    a.set_title(f"$k$ = {k}, $J$ = {m.inertia_:.0f}")
plotlib.save(f, "b1_voronoi")
slide("Voronoi cells")
print(
    "J falls with k whatever the truth: "
    + ", ".join(f"k={k}: {KMeans(k, n_init=10, random_state=SEED).fit(Xb).inertia_:.0f}" for k in (1, 2, 3, 6, 10))
)

## Slide: Four linkages on the same ten points

Agglomerative clustering starts with $n$ singletons and merges the two closest clusters until one is left. The
distance between clusters is the linkage: single (nearest pair), complete (farthest pair), average, Ward (growth
of the within-cluster sum of squares).

In [ ]:
rows = []
for link in ("single", "complete", "average", "ward"):
    Z = hc.linkage(P, link)
    rows.append((link, *Z[-3:, 2].round(2), str(hc.fcluster(Z, 2, "maxclust").tolist())))
slide("Four linkages on the same points")
show(
    pl.DataFrame(
        rows,
        schema=["linkage", "merge height n-3", "merge height n-2", "last merge height", "2 clusters"],
        orient="row",
    )
)
f, ax = plt.subplots(1, 4, figsize=(plotlib.FULL, 1.45), sharey=False)
for a, link in zip(ax, ("single", "complete", "average", "ward")):
    hc.dendrogram(
        hc.linkage(P, link), ax=a, color_threshold=0, above_threshold_color=GREY, no_labels=False, leaf_font_size=5.5
    )
    a.set_title(link)
    a.grid(False)
plotlib.save(f, "b1_dendrograms")

## Slide: Cut the tree at a height

One tree, every $k$: cutting at height $h$ gives the clusters whose merge height is below $h$. The long vertical
gap before the last merges is the evidence for the number of clusters.

In [ ]:
Xh, yh = B["blobs"]
Zw = hc.linkage(Xh, "ward")
f, ax = plt.subplots(1, 2, figsize=(plotlib.FULL, 1.65), gridspec_kw={"width_ratios": [1.5, 1]})
hc.dendrogram(
    Zw,
    ax=ax[0],
    truncate_mode="lastp",
    p=12,
    color_threshold=Zw[-3, 2] + 1e-9,
    leaf_font_size=5.5,
    above_threshold_color=GREY,
)
ax[0].axhline(Zw[-3, 2] * 0.98, color="k", ls="--", lw=0.7)
ax[0].grid(False), ax[0].set_ylabel("merge height")
cut = hc.fcluster(Zw, 3, "maxclust")
ax[1].scatter(*Xh.T, c=[COLS[t - 1] for t in cut], s=5, lw=0)
ax[1].set_xticks([]), ax[1].set_yticks([])
plotlib.save(f, "b1_cut")
slide("Cut the tree at a height")
print("last five merge heights (Ward):", Zw[-5:, 2].round(1), " ARI of the 3-cluster cut:", round(ari(yh, cut), 3))

## Slide: Determinism

Hierarchical clustering has no random start: the tree is a function of the data. Unseeded $k$-means with one
start per run changes with the seed. Ten runs of each on the `varied` set.

In [ ]:
Xv, yv = B["varied"]
hier = [AgglomerativeClustering(3, linkage="ward").fit_predict(Xv) for _ in range(10)]
same = all((h == hier[0]).all() for h in hier)
km_runs = [KMeans(3, init="random", n_init=1, random_state=s).fit(Xv) for s in range(10)]
slide("Determinism")
print(f"hierarchical Ward: {len(set(map(tuple, hier)))} distinct labelings in 10 runs, identical = {same}")
print(
    f"k-means, one random start: {len({round(m.inertia_, 3) for m in km_runs})} distinct costs; ARI to the truth ranges "
    f"{min(ari(yv, m.labels_) for m in km_runs):.3f} .. {max(ari(yv, m.labels_) for m in km_runs):.3f}"
)
pairs = [ari(km_runs[i].labels_, km_runs[j].labels_) for i in range(10) for j in range(i + 1, 10)]
print(f"mean ARI between two k-means runs {np.mean(pairs):.3f}; between two hierarchical runs 1.000")

## Slide: Ward is $k$-means with a tree; single linkage is DBSCAN with $\mathrm{minPts}=1$

Ward merges the pair whose merge raises $J$ the least, so its $k$-cut is a $k$-means-like partition. Single linkage
connects points closer than $h$: the connected components of the distance-$h$ graph, which are the DBSCAN
clusters when every point is a core point.

In [ ]:
rows = []
for name in ("blobs", "aniso", "varied", "moons", "circles"):
    Xs, ys = B[name]
    k = len(set(ys))
    ward = AgglomerativeClustering(k, linkage="ward").fit_predict(Xs)
    kmeans = KMeans(k, n_init=10, random_state=SEED).fit_predict(Xs)
    rows.append((name, ari(kmeans, ward), ari(ys, kmeans), ari(ys, ward)))
slide("Ward against k-means")
show(
    pl.DataFrame(rows, schema=["set", "ARI(k-means, Ward)", "ARI k-means vs truth", "ARI Ward vs truth"], orient="row")
)

Xm, ym = B["moons"]
eps = 0.2
Zs = hc.linkage(Xm, "single")
cut_single = hc.fcluster(Zs, eps, "distance")
db = DBSCAN(eps=eps, min_samples=1).fit(Xm)
slide("Single linkage is DBSCAN with minPts = 1")
print(
    f"moons, eps = {eps}: single-linkage cut gives {len(set(cut_single))} clusters, DBSCAN(min_samples=1) {len(set(db.labels_))}; "
    f"ARI between them {ari(cut_single, db.labels_):.3f}"
)
two = hc.fcluster(Zs, 2, "maxclust")
db5 = DBSCAN(eps=eps, min_samples=5).fit(Xm)
print(
    f"two-cluster single cut: ARI to the truth {ari(ym, two):.3f}; DBSCAN minPts=5 finds {len(set(db5.labels_)) - (-1 in db5.labels_)} clusters, noise {(db5.labels_ == -1).sum()} points"
)
f, ax = plt.subplots(1, 3, figsize=(plotlib.FULL, 1.45), sharex=True, sharey=True)
for a, lab, t in zip(
    ax,
    (KMeans(2, n_init=10, random_state=SEED).fit_predict(Xm), two, db5.labels_),
    ("k-means", "single linkage", "DBSCAN"),
):
    a.scatter(*Xm.T, c=[GREY if v < 0 else COLS[v % 7] for v in lab], s=5, lw=0)
    a.set_title(t), a.set_xticks([]), a.set_yticks([])
plotlib.save(f, "b1_moons")

## Sweep and failure: chaining and cost

Single linkage chains: one bridge of points joins two groups. Add a bridge of $m$ points between two blobs and
watch the single-linkage ARI collapse while Ward is unaffected. The cost of the tree: $O(n^2)$ memory.

In [ ]:
rng = np.random.default_rng(SEED)
Xa, ya = make_blobs(200, centers=[(-3, 0), (3, 0)], cluster_std=0.7, random_state=SEED)
rows = []
for m in (0, 2, 4, 8, 16):
    bridge = np.c_[np.linspace(-1.5, 1.5, m), rng.normal(0, 0.05, m)] if m else np.empty((0, 2))
    Xc2 = np.r_[Xa, bridge]
    yc = np.r_[ya, np.full(m, -1)]
    keep = yc >= 0
    s = AgglomerativeClustering(2, linkage="single").fit_predict(Xc2)
    w = AgglomerativeClustering(2, linkage="ward").fit_predict(Xc2)
    rows.append((m, ari(yc[keep], s[keep]), ari(yc[keep], w[keep])))
slide("Chaining")
show(pl.DataFrame(rows, schema=["bridge points", "ARI single", "ARI Ward"], orient="row"))

rows = []
for n in (500, 1000, 2000, 4000):
    Xt = np.random.default_rng(SEED).normal(size=(n, 2))
    t0 = time.perf_counter()
    KMeans(3, n_init=1, random_state=SEED).fit(Xt)
    tk = time.perf_counter() - t0
    t0 = time.perf_counter()
    AgglomerativeClustering(3, linkage="ward").fit(Xt)
    th = time.perf_counter() - t0
    rows.append((n, tk, th))
slide("Complexity")
show(pl.DataFrame(rows, schema=["n", "k-means seconds", "Ward seconds"], orient="row"), digits=4)

## Slide: The mean minimises the cost

The inertia of the first cluster of the ten points as a function of the x-coordinate of its centroid (y fixed at the
mean): a parabola whose minimum is the mean, $\mu_x = 1.6$.

In [ ]:
c0 = P[:5]
grid = np.linspace(0, 4, 200)
Jc = [((c0[:, 0] - g) ** 2).sum() + ((c0[:, 1] - c0[:, 1].mean()) ** 2).sum() for g in grid]
f, ax = plt.subplots(figsize=(1.6, 1.0))
ax.plot(grid, Jc, color=BLUE)
ax.axvline(c0[:, 0].mean(), color=RED, ls="--", lw=0.8)
ax.scatter(c0[:, 0], np.full(5, min(Jc)), marker="|", color="k", s=30, zorder=3)
ax.set_xlabel("centroid $x$"), ax.set_ylabel("cost of cluster 0")
plotlib.save(f, "b1_mean")
slide("The mean minimises the cost")
print(
    f"minimum of the parabola at x = {grid[int(np.argmin(Jc))]:.2f}, mean of the five x-values {c0[:, 0].mean():.2f}, cost {min(Jc):.2f}"
)